# One catalogue, two modalities

**Lecture 23 · Build** · Géron, Chapters 15–16

Applications of Machine Learning — BSc Mathematics of Artificial Intelligence

---

**How to use this notebook.** You are not expected to type the code. You are
expected to *read* it before you run it, and to be able to say what every line
does and what would break if it changed. Cells marked **⚠ read before running**
contain a defect on purpose.

**What it downloads.** A 5 MB index of the COCO validation split, then 200
individual images (about 30 MB), then three model checkpoints (about 1 GB in
total, cached after the first run). It does **not** download COCO, which is
about 20 GB.

**Expected wall clock on a Colab GPU runtime:** three to five minutes end to
end, most of it the first model download.

**About the prompt boxes.** Every code cell in this notebook is preceded by a
quoted prompt, and three lines follow it: what the prompt leaves open, the
version a student typically writes instead, and how you would catch a wrong
answer. Those three lines are the part worth reading twice.

The prompts here are **specifications, not transcripts** — this is what you
would have to ask for in order to get this cell, not a recording of somebody
asking for it. If your own prompt is vaguer than the box, expect worse code than
the cell below it.

*Lecture 19 is the one exception in this course.* It was built cell by cell
against Colab's Gemini 3.1 Pro, and its prompts are verbatim. It says so itself.


## 1 · Setup

> **Prompt · setup**
>
> **input** · nothing
>
> **output** · versions, seeds, device, and N_CATALOGUE
>
> **constraint** · `N_CATALOGUE = 200` as a named constant — every recall in this notebook is over that many candidates, and a recall without its candidate-set size is not a number

**Watch this prompt.**

* **Left open:** what is downloaded: a 5 MB split index, 200 images, and about 1 GB of model checkpoints. NOT COCO, which is about 20 GB.
* **The usual student version:** quoting 'Recall@1 = 34%' with no candidate count. Over 200 candidates and over 5,000 candidates those are entirely different claims.
* **How you would catch it:** the candidate-set size belongs in the same sentence as the recall, every time. Put it in a constant so the printout carries it.

In [ ]:
# --- setup -------------------------------------------------------------------
# Not examinable: version hygiene. It is here because a mismatch produces a
# confusing error twenty cells later rather than here.
import ast, io, sys, time, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from PIL import Image

print(f"python        {sys.version.split()[0]}")
print(f"torch         {torch.__version__}")
print(f"transformers  {transformers.__version__}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"device        {device}")

N_CATALOGUE = 200          # every recall below is over this many candidates

## 2 · The catalogue

Two downloads, and neither of them is COCO.

1. The **split index**: a CSV listing 5,000 COCO 2014 validation images with the
   five human-written captions each. About 5 MB.
2. The **images we actually use**: the first 200 by COCO id, fetched one at a
   time. About 30 MB.

Sorting by image id and taking the first 200 makes the catalogue deterministic —
two students with differently ordered frames get the same 200 images, for the
same reason `KFold(shuffle=True, random_state=42)` gives them the same folds.

**Expected wall clock: about one minute** the first time, instant afterwards.

> **Prompt · ⏱ 1 min first time — the catalogue**
>
> **input** · the COCO split index and the first 200 images by id
>
> **output** · 200 catalogue entries with their captions and files
>
> **constraint** · sort by `cocoid` and take the first 200 — two students with differently ordered frames then get the same 200 images, for the same reason a seeded KFold gives them the same folds
>
> **check** · assert the count, that every entry has at least two captions, and that the SKUs are unique

**Watch this prompt.**

* **Left open:** that this is a deterministic RULE and not a sample. Nobody chose which images make retrieval look good.
* **The usual student version:** `split.head(200)` on an unsorted frame, or `.sample(200)` without a seed. Both give 200 images and neither gives the same 200 twice.
* **How you would catch it:** normalise the caption whitespace on the way in. COCO's captions carry stray newlines, and a query that differs from a description only in whitespace is a bug you will chase later.

In [ ]:
CACHE = Path("datasets/app12")
CACHE.mkdir(parents=True, exist_ok=True)

CSV_URL = ("https://huggingface.co/datasets/nlphuji/"
           "mscoco_2014_5k_test_image_text_retrieval/resolve/main/"
           "test_5k_mscoco_2014.csv")

csv_path = CACHE / "coco_karpathy_test.csv"
if not csv_path.is_file():
    urllib.request.urlretrieve(CSV_URL, csv_path)

split = pd.read_csv(csv_path).sort_values("cocoid").reset_index(drop=True)
print(f"the split index lists {len(split):,} images")

imgdir = CACHE / "images"
imgdir.mkdir(exist_ok=True)

catalogue = []
for _, row in split.head(N_CATALOGUE).iterrows():
    dest = imgdir / row["filename"]
    if not dest.is_file():
        urllib.request.urlretrieve(
            f"http://images.cocodataset.org/val2014/{row['filename']}", dest)
    captions = [" ".join(c.split()) for c in ast.literal_eval(row["raw"])]
    catalogue.append({"sku": f"CAT-{int(row['cocoid']):06d}",
                      "file": dest, "captions": captions})

# assert, do not hope
assert len(catalogue) == N_CATALOGUE, len(catalogue)
assert all(len(e["captions"]) >= 2 for e in catalogue), "an entry has < 2 captions"
assert len({e["sku"] for e in catalogue}) == N_CATALOGUE, "duplicate SKU"

megabytes = sum(e["file"].stat().st_size for e in catalogue) / 1e6
print(f"catalogue: {len(catalogue)} entries, {megabytes:.1f} MB on disk")

### One entry

The five captions are **evidence**, not part of the product. We use caption #1 as
the entry's written description and caption #2 as the customer's query, so the
two sides are different people's sentences about the same picture. If we used
the same sentence on both sides a hash table would score 100% and the metric
would measure nothing.

> **Prompt · caption 1 is the description, caption 2 is the query**
>
> **input** · the catalogue entries
>
> **output** · the images, the descriptions and the queries as three parallel lists
>
> **constraint** · use DIFFERENT captions for the two sides — the five captions are evidence, not part of the product, and caption 1 as description with caption 2 as query means two different people's sentences about the same picture
>
> **check** · assert the three lists are the same length AND that the first description differs from the first query

**Watch this prompt.**

* **Left open:** what happens if you use the same sentence on both sides: a hash table scores 100% and the metric measures nothing at all.
* **The usual student version:** using caption 1 on both sides because it is one variable fewer. It is the first entry in the red-team list of retrieval leaks, and it is invisible in every number.
* **How you would catch it:** the assert that the two sides differ. One line, and it is the difference between measuring retrieval and measuring string equality.

In [ ]:
e = catalogue[0]
print(e["sku"])
for i, c in enumerate(e["captions"][:3], start=1):
    print(f"  caption {i}: {c}")

images       = [Image.open(x["file"]).convert("RGB") for x in catalogue]
descriptions = [x["captions"][0] for x in catalogue]
queries      = [x["captions"][1] for x in catalogue]

assert len(images) == len(descriptions) == len(queries) == N_CATALOGUE
assert descriptions[0] != queries[0], "query and description must differ"

## 3 · The metric, and the trivial baseline

The system returns a ranking, so the quantity that matters is the **rank of the
relevant entry**. Recall@k is the share of queries whose answer landed in the
top *k*.

Ties count **against** the model: a tie is scored as the worse rank. A metric
that flatters a degenerate score matrix is not a metric, and you will see why
three cells from now.

The trivial baseline needs no experiment at all. One relevant entry among *n*,
ranked uniformly at random, gives `P(rank <= k) = k/n` exactly.

> **Prompt · the metric, and the exact baseline**
>
> **input** · a square similarity matrix whose truth is the diagonal
>
> **output** · Recall@1, 5 and 10, the median rank, and the random-ranking values
>
> **constraint** · ties count AGAINST the model — use `>=`, not `>` — and compute the random baseline ARITHMETICALLY as k/n rather than by simulation
>
> **check** · assert the matrix is square before reading a diagonal out of it

**Watch this prompt.**

* **Left open:** why the tie rule matters, and the cell demonstrates it: with a strict `>` a model that outputs a constant reports R@1 = 100%.
* **The usual student version:** breaking ties in the model's favour, usually by accident via `argsort`. A degenerate score matrix then looks perfect, and this is exactly the failure mode of a badly initialised encoder.
* **How you would catch it:** a baseline you can compute in closed form beats a simulated one. One relevant entry among n, ranked uniformly, gives P(rank ≤ k) = k/n exactly — no seeds, no noise.

In [ ]:
def ranks_of_truth(sim):
    """sim[i, j] = score of query i against candidate j; truth is j == i."""
    assert sim.shape[0] == sim.shape[1], sim.shape
    correct = np.diag(sim)[:, None]
    return (sim >= correct).sum(axis=1)          # 1 = best; ties count against us


def report(name, sim):
    r = ranks_of_truth(sim)
    n = len(r)
    out = {f"R@{k}": (r <= k).mean() for k in (1, 5, 10)}
    print(f"{name:34s} " + "  ".join(f"{k} {v:6.1%}" for k, v in out.items())
          + f"   median rank {np.median(r):5.0f} of {n}")
    return out


n = N_CATALOGUE
print(f"random ranking over {n} candidates, computed exactly:")
for k in (1, 5, 10):
    print(f"  R@{k:<3d} {k / n:6.1%}")
print(f"  expected rank {(n + 1) / 2:.1f}")
print(f"  MRR {np.mean(1 / np.arange(1, n + 1)):.3f}")

# the tie rule, demonstrated: a model that has learned nothing
zeros = np.zeros((n, n))
strict = 1 + (zeros > np.diag(zeros)[:, None]).sum(axis=1)
print(f"\nwith a strict '>' a constant scorer reports R@1 = {(strict == 1).mean():.0%}")
print(f"with '>=' it reports R@1 = {(ranks_of_truth(zeros) <= 1).mean():.1%}")

### ✍️ Commit

Before running another cell, write down on paper:

* the metric, and the candidate-set size;
* the Recall@1 a good system should reach;
* the Recall@1 you expect from what we are about to build.

Shuffling scores 0.5% at rank 1 and 5.0% in the top ten over 200 candidates.
Your number belongs somewhere above those, and saying *where* is the exercise.

## 4 · Encode the images

An image-only encoder: a vision transformer trained on ImageNet labels. No text
appears anywhere in its training.

**Expected wall clock: 20–60 s** for the model download, then about 15 s of
inference for 200 images.

> **Prompt · ⏱ 20-60 s — encode the images**
>
> **input** · the 200 catalogue images
>
> **output** · 768-dimensional CLS vectors
>
> **constraint** · `add_pooling_layer=False` — that checkpoint carries no pooler weights, so asking for one gives you a RANDOMLY INITIALISED layer with no error and no warning
>
> **check** · assert the shape is (200, 768)

**Watch this prompt.**

* **Left open:** that no text appears anywhere in this model's training. It is a vision transformer trained on ImageNet labels, and that fact is the whole of section 7.
* **The usual student version:** using the pooler output, which on this checkpoint is a random projection of the CLS token. Everything runs and the geometry is noise.
* **How you would catch it:** when a checkpoint warns that weights were newly initialised, read it. It is the single most-ignored message in the transformers library.

In [ ]:
from transformers import AutoImageProcessor, ViTModel

vit_proc = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
vit = ViTModel.from_pretrained("google/vit-base-patch16-224",
                               add_pooling_layer=False).to(device).eval()
# add_pooling_layer=False because that checkpoint carries no pooler weights.
# Ask for one and you get a randomly initialised layer, with no error.

t0 = time.perf_counter()
chunks = []
with torch.no_grad():
    for i in range(0, len(images), 32):
        batch = vit_proc(images=images[i:i + 32], return_tensors="pt").to(device)
        chunks.append(vit(**batch).last_hidden_state[:, 0].cpu().numpy())

V = np.concatenate(chunks).astype(np.float64)
assert V.shape == (N_CATALOGUE, 768), V.shape
print(f"images encoded: {V.shape} in {time.perf_counter() - t0:.1f}s")

## 5 · Encode the queries

A text-only encoder: mean-pooled sentence embeddings. No image appears anywhere
in its training. Note the attention mask in the pooling — forget it and you
average in the padding.

> **Prompt · encode the queries**
>
> **input** · the 200 query sentences
>
> **output** · 384-dimensional mean-pooled embeddings
>
> **constraint** · pool with the ATTENTION MASK — forget it and you average in the padding, and longer sentences are diluted more than short ones
>
> **check** · assert the shape is (200, 384)

**Watch this prompt.**

* **Left open:** that no image appears anywhere in this model's training either. Two encoders, two worlds, no contact.
* **The usual student version:** `last_hidden_state.mean(1)`, which is the padding bug from the previous application in a third costume.
* **How you would catch it:** any mean over a padded sequence needs the mask. If your pooling line has no `attention_mask` in it, it is wrong.

In [ ]:
from transformers import AutoModel, AutoTokenizer

TEXT_ID = "sentence-transformers/all-MiniLM-L6-v2"
tok = AutoTokenizer.from_pretrained(TEXT_ID)
enc = AutoModel.from_pretrained(TEXT_ID).to(device).eval()


def minilm(sentences, batch=64):
    out = []
    with torch.no_grad():
        for i in range(0, len(sentences), batch):
            b = tok(sentences[i:i + batch], return_tensors="pt", padding=True,
                    truncation=True, max_length=128).to(device)
            h = enc(**b).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).float()
            out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    return np.concatenate(out).astype(np.float64)


T = minilm(queries)
assert T.shape == (N_CATALOGUE, 384), T.shape
print(f"queries encoded: {T.shape}")

## 6 · Now take the cosine

Reviewer question 3: **what is the shape here?**

> **Prompt · reviewer question 3 — what is the shape here**
>
> **input** · the two feature matrices
>
> **output** · their shapes, and the ValueError from multiplying them
>
> **constraint** · catch the error and PRINT it rather than letting the notebook stop — the exception is the content of the cell

**Watch this prompt.**

* **Left open:** the sentence the error is really saying: there is no cosine between them, because there is no inner product between them.
* **The usual student version:** reaching for a projection immediately, which is the next cell and which does not fix anything. The shape mismatch is a symptom of something the shapes cannot express.
* **How you would catch it:** 768 and 384 is a loud failure. The quiet version is two encoders that happen to share a dimension, where the multiplication succeeds and means nothing.

In [ ]:
print(f"V {V.shape}   T {T.shape}")
try:
    V @ T.T
except ValueError as exc:
    print(f"\nValueError: {exc}")
    print("\nThere is no cosine between them, because there is no inner "
          "product between them.")

### Force the dimensions to agree — three ways

So nobody can blame the fix:

1. a random Gaussian projection 768 → 384 (Johnson–Lindenstrauss, thread 5);
2. keep the first 384 coordinates of the image vector;
3. zero-pad the text vector to 768.

Thread 5 promised that a random projection nearly preserves the geometry *of the
image space*. It promised nothing about aligning that geometry with anything
else.

> **Prompt · force the dimensions to agree — three ways**
>
> **input** · the two feature matrices
>
> **output** · Recall@1, 5, 10 for a JL projection, a truncation and a zero-pad
>
> **constraint** · do it THREE ways, so nobody can blame the particular fix
>
> **check** · assert each similarity matrix is square and 200 by 200 before reporting it

**Watch this prompt.**

* **Left open:** what the JL theorem actually promised. A random projection nearly preserves the geometry OF THE IMAGE SPACE. It promised nothing about aligning that geometry with anything else.
* **The usual student version:** trying one projection, getting chance, and concluding the projection was badly chosen. All three land at chance and that is the argument.
* **How you would catch it:** transpose before reporting. `report` expects rows to be queries, and the matrices here are built image-major — a silent transpose measures image-to-text and calls it text-to-image.

In [ ]:
def unit(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)


rng = np.random.default_rng(SEED)
R = rng.normal(0, 1 / np.sqrt(384), size=(768, 384))

Vn, Tn = unit(V), unit(T)
sims_two_spaces = {
    "JL projection 768->384": unit(Vn @ R) @ Tn.T,
    "truncate image to 384":  unit(Vn[:, :384]) @ Tn.T,
    "zero-pad text to 768":   Vn @ unit(np.pad(Tn, ((0, 0), (0, 768 - 384)))).T,
}
for name, S in sims_two_spaces.items():
    assert S.shape == (N_CATALOGUE, N_CATALOGUE), (name, S.shape)
    report(name, S.T)                      # transpose: rows are text queries
print(f"\nrandom ranking over {n} candidates: R@1 {1 / n:.1%}  "
      f"R@5 {5 / n:.1%}  R@10 {10 / n:.1%}")

## 7 · ⚠ Read before running — the assistant failure

**The prompt:** *"Encode the catalogue images with a ViT and the captions with a
sentence transformer, then check whether an image and its caption are similar."*

Under-specified in exactly one place. Find it before you run the cell.

> **Prompt · ⚠ what the weak prompt returns**
>
> **input** · 'encode the images with a ViT and the captions with a sentence transformer, then check whether an image and its caption are similar'
>
> **output** · the mean cosine of each matched pair, and the share above 0.05
>
> **constraint** · compute only the DIAGONAL, as the prompt implies — this is the failure, not the fix

**Watch this prompt.**

* **Left open:** the review question: what does an UNRELATED pair score? This cell computed 200 numbers and never touched the other 39,800.
* **The usual student version:** reading 'mean similarity 0.14' as evidence of anything. It reports a LEVEL where only a DIFFERENCE means anything.
* **How you would catch it:** reviewer question 5, in an unusual form. The default nobody asked for here is the missing control group.

In [ ]:
# --- what the weak prompt returns --------------------------------------------
Vp = unit(unit(V) @ R)
sim_matched = (Vp * unit(T)).sum(axis=1)      # cosine of each matched pair

print(f"mean similarity of an image and its caption: {sim_matched.mean():.3f}")
print(f"{(sim_matched > 0.05).mean():.0%} of pairs score above 0.05")

**The review question:** *what does an unrelated pair score?*

The cell computed the diagonal of the similarity matrix and never touched the
other 200 × 200 − 200 = 39,800 entries. It reports a **level** where only a
**difference** means anything. In the five reviewer questions this is number 5 —
the default nobody asked for is the missing control group.

> **Prompt · the control that was missing**
>
> **input** · the full 200 × 200 similarity matrix
>
> **output** · the mean and sd of the diagonal and off-diagonal, and Cohen's d
>
> **constraint** · report a STANDARDISED difference — the raw gap between two means is unreadable without the spread they are drawn from

**Watch this prompt.**

* **Left open:** why it happened. Both encoders work: the ViT separates images from images, MiniLM separates sentences from sentences. Neither ever saw a single (image, sentence) pair, so neither has any reason to place them anywhere in particular relative to each other.
* **The usual student version:** concluding the models are broken. Apply any rotation to the image space and every image-image cosine is unchanged, so the ViT's loss is unchanged, and every image-text cosine changes. The loss cannot tell those worlds apart, so it did not pick one.
* **How you would catch it:** `ddof=1` on both variances, and n printed beside each. 200 against 39,800 is a very asymmetric comparison and the reader should see it.

In [ ]:
S = sims_two_spaces["JL projection 768->384"]
off = ~np.eye(N_CATALOGUE, dtype=bool)
matched, unrelated = np.diag(S), S[off]

pooled = np.sqrt((matched.var(ddof=1) + unrelated.var(ddof=1)) / 2)
print(f"matched pairs    mean {matched.mean():+.4f}   sd {matched.std(ddof=1):.4f}"
      f"   (n = {len(matched)})")
print(f"unrelated pairs  mean {unrelated.mean():+.4f}   sd {unrelated.std(ddof=1):.4f}"
      f"   (n = {len(unrelated):,})")
print(f"difference       {matched.mean() - unrelated.mean():+.4f}"
      f"   Cohen's d {(matched.mean() - unrelated.mean()) / pooled:+.3f}")

**The corrected specification:**

> Build the full 200 × 200 cosine matrix. Report the mean and sd of the diagonal
> **and** of the off-diagonal, their standardised difference, and Recall@1, 5
> and 10 for text-to-image ranking. Print the exact random-ranking values `k/n`
> beside them. Break ties against the model.

Every clause is there because leaving it out produced a number somebody
believed.

---

### Why it happened

Both encoders work. The ViT separates images from images; MiniLM separates
sentences from sentences. Neither ever saw a single (image, sentence) pair, so
neither has any reason to place them anywhere in particular relative to each
other.

Apply any rotation to the image space: every image–image cosine is unchanged, so
the ViT's loss is unchanged, and every image–text cosine changes. The loss
cannot tell those worlds apart, so it did not pick one.

## 8 · A jointly trained pair

Same architecture family, one difference: both towers were trained by a single
objective that compared them. **Expected wall clock: 1–2 min** for the download
(about 600 MB), then a few seconds of inference.

> **Prompt · ⏱ 1-2 min — a jointly trained pair**
>
> **input** · the same images and queries
>
> **output** · image and text features in a shared 512-dimensional space
>
> **constraint** · `unit()` BOTH sides — `get_image_features` and `get_text_features` return vectors that are NOT normalised
>
> **check** · assert both matrices have the same shape and the model's own projection dimension

**Watch this prompt.**

* **Left open:** that skipping the normalisation still runs, still returns a matrix, and still produces a ranking — a different one. That is the next lecture's assistant failure, and it costs more than you would guess.
* **The usual student version:** assuming a model that returns 'features' returns unit vectors. Most do not, and the cosine of unnormalised vectors is a dot product weighted by two arbitrary magnitudes.
* **How you would catch it:** one difference from section 4: both towers were trained by a single objective that COMPARED them. Same architecture family, one difference, and it is the whole result.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = "openai/clip-vit-base-patch32"
clip_proc = CLIPProcessor.from_pretrained(CLIP_ID)
clip = CLIPModel.from_pretrained(CLIP_ID).to(device).eval()

print(f"projection dim {clip.config.projection_dim}")
print(f"parameters     {sum(p.numel() for p in clip.parameters()) / 1e6:.0f}M")


def clip_images(imgs, batch=32):
    out = []
    with torch.no_grad():
        for i in range(0, len(imgs), batch):
            b = clip_proc(images=imgs[i:i + batch], return_tensors="pt").to(device)
            out.append(clip.get_image_features(**b).cpu().numpy())
    return np.concatenate(out).astype(np.float64)


def clip_text(sentences, batch=64):
    out = []
    with torch.no_grad():
        for i in range(0, len(sentences), batch):
            b = clip_proc(text=sentences[i:i + batch], return_tensors="pt",
                          padding=True, truncation=True, max_length=77).to(device)
            out.append(clip.get_text_features(**b).cpu().numpy())
    return np.concatenate(out).astype(np.float64)


I_raw = clip_images(images)
Q_raw = clip_text(queries)
assert I_raw.shape == Q_raw.shape == (N_CATALOGUE, clip.config.projection_dim)

I, Q = unit(I_raw), unit(Q_raw)      # onto the sphere — both sides, always
print(f"\nimage features {I.shape}   text features {Q.shape}")

`get_image_features` and `get_text_features` return vectors that are **not**
normalised. Skipping `unit()` still runs, still returns a matrix, and still
produces a ranking — a different one. That is the assistant failure of the next
lecture, and it costs more than you would guess.

## 9 · The known-answer test, before the real one

Step 4 of the working loop: *test against a case whose answer you know*. The
catalogue has no labels, so nothing in it can tell us the model is loaded
correctly and preprocessing its inputs the way it was trained to.

CIFAR-10 can. **Expected wall clock: 1–2 min** for the 170 MB download the first
time, then about 10 s of inference for 500 images.

> **Prompt · ⏱ 1-2 min — a known-answer test first**
>
> **input** · 500 CIFAR-10 test images
>
> **output** · their CLIP features
>
> **constraint** · use a dataset WITH LABELS — the catalogue has none, so nothing in it can tell us the model is loaded correctly and preprocessing its inputs the way it was trained to
>
> **check** · assert the counts and that there are ten classes

**Watch this prompt.**

* **Left open:** that this is step 4 of the working loop: test against a case whose answer you know. It is not about CIFAR and it is not about zero-shot classification.
* **The usual student version:** going straight to the retrieval numbers. If the processor is mismatched to the checkpoint, retrieval degrades gracefully and silently, and nothing in the catalogue can catch it.
* **How you would catch it:** say `over 500 images` wherever you quote the accuracy. A subset is a subset even when the point of the cell is not the number.

In [ ]:
from torchvision.datasets import CIFAR10

cifar = CIFAR10(root="datasets", train=False, download=True)
N_CIFAR = 500                      # a subset, and we say so wherever we quote it
cifar_images = [cifar[i][0].convert("RGB") for i in range(N_CIFAR)]
y = np.array([cifar[i][1] for i in range(N_CIFAR)])
classes = list(cifar.classes)

assert len(cifar_images) == len(y) == N_CIFAR
assert len(classes) == 10, classes

F = unit(clip_images(cifar_images))
print(f"{N_CIFAR} CIFAR-10 images encoded: {F.shape}")

> **Prompt · the classifier is ten sentences**
>
> **input** · three prompt templates
>
> **output** · zero-shot accuracy under each, against chance
>
> **constraint** · try SEVERAL templates and print all of them — a single template hides that the number depends on it

**Watch this prompt.**

* **Left open:** that there is no fitted parameter anywhere in this cell. The classifier is ten sentences, and the sentence is a choice you made.
* **The usual student version:** reporting the best template's accuracy. Choosing the template by looking at the test accuracy is a hyperparameter chosen on the test set — application 3, in a new costume.
* **How you would catch it:** a 'zero-shot' number is a number for ONE PARTICULAR SENTENCE. The spread across templates is part of the result.

In [ ]:
for template in ["{}", "a photo of a {}", "a low-resolution photo of a {}"]:
    W = unit(clip_text([template.format(c) for c in classes]))
    pred = (F @ W.T).argmax(axis=1)
    print(f"{template.format('dog')!r:38s} accuracy {(pred == y).mean():6.1%}"
          f"   over {N_CIFAR} images   (chance 10.0%)")

The classifier is ten sentences; there is no fitted parameter anywhere in that
cell. But notice the spread across templates. A "zero-shot" number is a number
for **one particular sentence**, and the sentence is a choice you made.

Choosing the best template by looking at the test accuracy is a hyperparameter
chosen on the test set — Lecture 6, in a new costume.

## 10 · Back to the catalogue

Same 200 images, same 200 queries, same metric, same tie rule, same baseline.
Only the encoders changed.

> **Prompt · back to the catalogue**
>
> **input** · the CLIP features
>
> **output** · recall for the two-space route and both directions of the joint one, with the arithmetic baseline
>
> **constraint** · same 200 images, same 200 queries, same metric, same tie rule, same baseline — ONLY the encoders changed
>
> **check** · assert the similarity matrix is square

**Watch this prompt.**

* **Left open:** that text-to-image and image-to-text are different numbers on the same matrix. Reporting one of them as 'retrieval accuracy' hides which direction was measured.
* **The usual student version:** rebuilding the metric for the new model, or changing the tie rule, and then comparing. One variable.
* **How you would catch it:** print the random baseline in the same table, every time. It is the only thing that makes 34% legible.

In [ ]:
sim_clip = Q @ I.T                        # rows: text queries, columns: images
assert sim_clip.shape == (N_CATALOGUE, N_CATALOGUE)

print(f"over {N_CATALOGUE} candidates, ties against the model:\n")
report("ViT + MiniLM, JL projection",
       sims_two_spaces["JL projection 768->384"].T)
report("joint dual encoder, text->image", sim_clip)
report("joint dual encoder, image->text", I @ Q.T)
print(f"\n{'random ranking (arithmetic)':34s} " +
      "  ".join(f"R@{k} {k / n:6.1%}" for k in (1, 5, 10)) +
      f"   expected rank {(n + 1) / 2:5.0f} of {n}")

> **Prompt · the same control, on the joint model**
>
> **input** · the CLIP similarity matrix
>
> **output** · matched against unrelated, and Cohen's d
>
> **constraint** · run the IDENTICAL analysis as section 7 — the comparison is between two values of d, and it is only a comparison if the computation is the same

**Watch this prompt.**

* **Left open:** the size of the difference. Section 7's d was near zero; this one is not, and that gap is the entire application in one number.
* **The usual student version:** reporting the recall improvement alone. The effect size says something the recall cannot: that the two distributions have separated rather than that one ranking got luckier.
* **How you would catch it:** when you fix something, re-run the diagnostic that detected it, unchanged. A new diagnostic on the fixed version proves nothing about the old one.

In [ ]:
matched, unrelated = np.diag(sim_clip), sim_clip[~np.eye(N_CATALOGUE, dtype=bool)]
pooled = np.sqrt((matched.var(ddof=1) + unrelated.var(ddof=1)) / 2)
print(f"matched pairs    mean {matched.mean():+.4f}   sd {matched.std(ddof=1):.4f}")
print(f"unrelated pairs  mean {unrelated.mean():+.4f}   sd {unrelated.std(ddof=1):.4f}")
print(f"difference       {matched.mean() - unrelated.mean():+.4f}"
      f"   Cohen's d {(matched.mean() - unrelated.mean()) / pooled:.2f}")

## 11 · The other route, and where it breaks

Some entries have a written description. For those, the previous application's
semantic search applies directly: embed the description, embed the query, rank by
cosine. Same space on both sides, because both sides are sentences.

Then model the catalogue as it really arrives: delete the description of 60 of
the 200 entries, by a fixed rule so the experiment repeats.

> **Prompt · the other route, and where it breaks**
>
> **input** · the written descriptions, with 60 of them deleted by a fixed rule
>
> **output** · recall with and without the descriptions, on the affected entries and via the image route
>
> **constraint** · delete by a FIXED RULE (`i % 10 < 3`) so the experiment repeats, and set the deleted columns to −inf so they are not in the index at all
>
> **check** · assert exactly 60 entries were blanked

**Watch this prompt.**

* **Left open:** that the answer is not 'worse'. It is ZERO, structurally — an entry with no text is not in a text index, and no amount of better embedding changes that.
* **The usual student version:** imputing an empty string for the missing descriptions, which puts them in the index at some arbitrary point and turns a structural zero into a small number that looks like a modelling result.
* **How you would catch it:** the image route is untouched on exactly those 60 entries, because it never read a description. That contrast is the number the next lecture repairs.

In [ ]:
D = unit(minilm(descriptions))
Qt = unit(minilm(queries))
sim_text = Qt @ D.T
report("text query -> text description", sim_text)

blanked = np.array([i for i in range(N_CATALOGUE) if i % 10 < 3])
assert len(blanked) == 60, len(blanked)

sim_missing = sim_text.copy()
sim_missing[:, blanked] = -np.inf          # not in the index at all
report("...with 60 descriptions deleted", sim_missing)

r_full    = ranks_of_truth(sim_text)
r_missing = ranks_of_truth(sim_missing)
print(f"\nR@1 on those 60 entries: with a description "
      f"{(r_full[blanked] <= 1).mean():.1%}, with none "
      f"{(r_missing[blanked] <= 1).mean():.1%}")
print(f"R@1 on the same 60 via the joint image route: "
      f"{(ranks_of_truth(sim_clip)[blanked] <= 1).mean():.1%}")

Not "worse" — **zero, structurally**. An entry with no text is not in a text
index. The image route is untouched, because it never read a description.

That zero is the number the next lecture repairs.

## 12 · Red-team

Swap notebooks with the team beside you. Fifteen minutes. Five questions:

1. What touched the test set?
2. What was fitted, and on what? (`fit` and `transform` are different verbs)
3. What is the shape here?
4. What was dropped — rows, columns, NaNs? Count them.
5. What is the default I did not ask for?

And six ways to leak in a *retrieval* evaluation specifically:

1. the query text is also the indexed description — you are measuring string
   equality;
2. the shortlist a query is scored against was built using that query;
3. the prompt template was chosen by looking at the number it produced;
4. recall reported without the candidate-set size;
5. ties broken in the model's favour (see section 3 — a constant scorer then
   reports 100%);
6. normalisation applied to one side only, so "cosine" is not a cosine.

Report what you **found**, not what you would have done differently.

---

### ✍️ Before you leave

Add to your sheet: the best Recall@1 you obtained, over how many candidates, and
one query the system got wrong together with your explanation of why.